In [ ]:
%%writefile tokenizer.py
"""
Shared tokenizer + vocabulary module for NLP Assignment 2, Project 1 (DailyDialog).

Used identically by Task 1 (n-grams), Task 2 (AWD-LSTM), Task 3 (hybrid), and the
BLiMP evaluation harness. Build the vocab ONCE from DailyDialog's train split and
reuse the same Vocab object everywhere — never rebuild it per-task.

Design choices (see PLAN.md Section 3 for the full rationale):
- Punctuation-aware whitespace tokenizer: splits words from attached punctuation
  (e.g. "hello!" -> ["hello", "!"]) but does not split contractions internally
  (e.g. "don't" stays as ["don't"]) since the spec doesn't ask for that level of
  linguistic splitting and DailyDialog is casual/contracted speech.
- Lowercase everything.
- Vocabulary built from train only, capped at 30,000 most frequent tokens.
  Tokens occurring <2 times in train, and any OOV token seen later, map to <UNK>.
- Special tokens: <unk>, <s>, </s>, <pad> (pad reserved for Task 2 batching; not
  used by Task 1's counting).
"""

import re
import json
from collections import Counter
from typing import List, Dict, Iterable


# ---------------------------------------------------------------------------
# Tokenization
# ---------------------------------------------------------------------------

# Matches: contractions/words with internal apostrophes (don't, it's) as one
# token, plain word tokens, or any single punctuation/symbol character.
_TOKEN_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?|[0-9]+(?:\.[0-9]+)?|[^\sA-Za-z0-9]")


def _normalize_apostrophes(text: str) -> str:
    """Collapse all Unicode apostrophe/quote variants to a single straight
    ASCII apostrophe (') before any other processing.

    Real DailyDialog text mixes the straight apostrophe (U+0027, ') with the
    Unicode "right single quotation mark" (U+2019, ’) and occasionally other
    look-alikes, inconsistently -- sometimes within the same split. Left
    unnormalized, contraction detection silently fails for every curly-quote
    instance (e.g. "I \u2019 m here" would NOT be caught by the spacing-fix
    regex below, since that regex only matches the ASCII character), leaking
    junk single-character tokens ('m', 't', 's', etc.) into the vocabulary.
    This must run BEFORE `_normalize_spacing`.
    """
    return text.replace("\u2019", "'").replace("\u2018", "'").replace("`", "'")


def _normalize_spacing(text: str) -> str:
    """Fix known DailyDialog spacing artifacts before tokenizing.

    Real DailyDialog utterances frequently have spaces on BOTH sides of the
    apostrophe inside a contraction, e.g. "Let ' s go" instead of "Let's go".
    Left alone, this breaks contraction detection entirely: the word, the
    apostrophe, and the suffix end up as three spurious separate tokens
    ("let", "'", "s") instead of one "let's" token, polluting the vocabulary
    with junk single-character tokens. This regex collapses "<word> ' <sfx>"
    into "<word>'<sfx>" whenever the trailing part looks like a common
    contraction suffix (s, t, re, ve, ll, d, m), so "let ' s" -> "let's",
    "they ' re" -> "they're", etc. Requires a preceding word character so it
    doesn't fire on a bare leading apostrophe/quote mark.

    This is a documented, deliberate preprocessing deviation -- state it in
    the report's preprocessing subsection.
    """
    text = re.sub(
        r"(\w)\s*'\s+(s|t|re|ve|ll|d|m)\b",
        r"\1'\2",
        text,
        flags=re.IGNORECASE,
    )
    return text


def tokenize(text: str) -> List[str]:
    """Punctuation-aware whitespace tokenizer + lowercase.

    Applies DailyDialog-specific spacing normalization (see
    `_normalize_spacing`) before the regex tokenizer runs.

    Example:
        >>> tokenize("Hello! Don't you think it's 3.5 degrees out?")
        ['hello', '!', "don't", 'you', 'think', "it's", '3.5', 'degrees', 'out', '?']
        >>> tokenize("Good . Let ' s go now .")
        ['good', '.', "let's", 'go', 'now', '.']
        >>> tokenize("I \u2019 m here .")
        ["i'm", 'here', '.']
    """
    text = _normalize_apostrophes(text)
    text = _normalize_spacing(text)
    text = text.lower()
    return _TOKEN_RE.findall(text)


def join_dialogue(utterances: List[str]) -> str:
    """Join a DailyDialog `dialog` list into one document.

    Per spec: concatenate utterances in order, separated by a single space.
    NO speaker tags, no turn markers.
    """
    return " ".join(u.strip() for u in utterances)


# ---------------------------------------------------------------------------
# Vocabulary
# ---------------------------------------------------------------------------

UNK = "<unk>"
BOS = "<s>"
EOS = "</s>"
PAD = "<pad>"

SPECIAL_TOKENS = [PAD, UNK, BOS, EOS]  # fixed order -> fixed ids 0,1,2,3


class Vocab:
    """Fixed vocabulary built from a token frequency count.

    Attributes:
        token2id: dict token -> int id
        id2token: list, index = id
        freq: Counter of raw training-set token frequencies (pre-cap, pre-UNK
              collapse) -- kept for diagnostics / report stats, not used at
              runtime.
    """

    def __init__(self, token2id: Dict[str, int], id2token: List[str], freq: Counter):
        self.token2id = token2id
        self.id2token = id2token
        self.freq = freq

    def __len__(self):
        return len(self.id2token)

    def encode(self, tokens: Iterable[str]) -> List[int]:
        unk_id = self.token2id[UNK]
        return [self.token2id.get(t, unk_id) for t in tokens]

    def encode_str(self, tokens: Iterable[str]) -> List[str]:
        """Map OOV tokens to the literal '<unk>' string (used by n-gram code
        that works with token strings rather than ids)."""
        vocab_set = set(self.token2id.keys())
        return [t if t in vocab_set else UNK for t in tokens]

    def decode(self, ids: Iterable[int]) -> List[str]:
        return [self.id2token[i] for i in ids]

    def save(self, path: str):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "id2token": self.id2token,
                    "freq": dict(self.freq),
                },
                f,
                ensure_ascii=False,
                indent=1,
            )

    @classmethod
    def load(cls, path: str) -> "Vocab":
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)
        id2token = obj["id2token"]
        token2id = {t: i for i, t in enumerate(id2token)}
        freq = Counter(obj["freq"])
        return cls(token2id, id2token, freq)


def build_vocab(
    train_token_lists: Iterable[List[str]],
    max_size: int = 30000,
    min_count: int = 2,
) -> Vocab:
    """Build a Vocab from an iterable of already-tokenized training documents.

    Args:
        train_token_lists: iterable where each element is the token list for
            one training document (one DailyDialog dialogue, already joined +
            tokenized). Pass the full train split here, nothing held out.
        max_size: cap on vocabulary size INCLUDING special tokens. Spec says
            "30,000 most frequent tokens" -- we interpret this as 30,000
            regular (non-special) tokens, with the 4 special tokens added on
            top, since the special tokens are not part of the natural-language
            frequency count the spec is capping. State this in the report if
            asked; trivial to change `max_size` semantics here if TA disagrees.
        min_count: tokens occurring fewer than this many times in train are
            collapsed into <UNK> even if they'd otherwise fit under max_size.

    Returns:
        Vocab object. Deterministic tie-breaking: when multiple tokens have
        identical frequency at the cutoff boundary, ties are broken
        alphabetically (ascending) for full reproducibility across runs/seeds.
    """
    freq = Counter()
    for tokens in train_token_lists:
        freq.update(tokens)

    # Keep tokens meeting min_count, then cap to max_size most frequent,
    # alphabetical tie-break for determinism.
    eligible = [(tok, c) for tok, c in freq.items() if c >= min_count]
    eligible.sort(key=lambda x: (-x[1], x[0]))  # freq desc, then alpha asc
    kept = [tok for tok, _ in eligible[:max_size]]

    id2token = list(SPECIAL_TOKENS) + kept
    token2id = {t: i for i, t in enumerate(id2token)}

    return Vocab(token2id, id2token, freq)


# ---------------------------------------------------------------------------
# Scoring-stream builder (shared by Task 1 n-gram eval AND Task 2 LSTM eval,
# so Task 3's per-position interpolation is well-defined -- see PLAN.md 3.7)
# ---------------------------------------------------------------------------

def build_scoring_stream(
    dialogues: List[List[str]],
    vocab: Vocab,
) -> List[str]:
    """Turn a list of dialogues (each a list of raw utterance strings) into
    ONE flat list of vocab-mapped token strings, with <s> ... </s> wrapped
    around each dialogue (per-dialogue boundaries, NOT per-utterance -- see
    PLAN.md assumption 3.2).

    This exact function must be used by both the n-gram scorer and the LSTM
    scorer at evaluation time so both models score an identical sequence of
    positions. Do not reimplement this logic separately in task2/task3 code.
    """
    stream: List[str] = []
    for utterances in dialogues:
        doc = join_dialogue(utterances)
        toks = tokenize(doc)
        toks = vocab.encode_str(toks)
        stream.append(BOS)
        stream.extend(toks)
        stream.append(EOS)
    return stream


if __name__ == "__main__":
    # Minimal smoke test using fabricated data (NOT real DailyDialog -- just
    # verifies the module's internal logic before we touch Colab/real data).
    fake_dialogues = [
        ["Hello there!", "Hi, how are you?", "I'm fine, thanks."],
        ["Don't forget the meeting.", "I won't. See you at 3.5pm!"],
        ["Hello there!", "Hi, how are you?"],  # repeat to test frequency counts
    ]

    tokenized_docs = [tokenize(join_dialogue(d)) for d in fake_dialogues]
    for d, toks in zip(fake_dialogues, tokenized_docs):
        print(join_dialogue(d), "->", toks)

    vocab = build_vocab(tokenized_docs, max_size=30000, min_count=2)
    print("\nVocab size (incl. specials):", len(vocab))
    print("Tokens kept:", [t for t in vocab.id2token if t not in SPECIAL_TOKENS])
    print("'hello' in vocab:", "hello" in vocab.token2id)
    print("'meeting' in vocab (freq 1, should be UNK'd):", "meeting" in vocab.token2id)

    stream = build_scoring_stream(fake_dialogues, vocab)
    print("\nScoring stream:")
    print(stream)

    # Save/load round trip
    vocab.save("/tmp/test_vocab.json")
    reloaded = Vocab.load("/tmp/test_vocab.json")
    assert reloaded.id2token == vocab.id2token
    print("\nSave/load round-trip OK.")

Overwriting tokenizer.py


In [ ]:
%%writefile ngram_lm.py
"""
N-gram language model: count tables + 5 smoothing variants (Task 1).

Shared counting + 3 smoothing families:
  - Add-k (k in {1, 0.5, 0.1})  -- 3 variants, same formula, different k
  - Good-Turing (r <= 5 re-estimated, backoff for r=0 -- see PLAN.md 3.3)
  - Interpolated Kneser-Ney (d=0.75 fixed, + outer lambda blend w/ unigram,
    lambda grid-searched on validation -- see PLAN.md 3.4)

Design: build ONE set of n-gram count tables per order (1..4) from the train
scoring-stream, then every smoothing method reads from those same tables.
Never recount per smoothing variant.

All models expose the same two methods so Task 1's PPL loop, Task 3's hybrid,
and the BLiMP scorer can all treat them interchangeably:
  - logprob(context: tuple[str,...], word: str) -> float   (natural log)
  - a `order` attribute (the n in n-gram, e.g. 3 for trigram)
"""

import math
from collections import defaultdict, Counter
from typing import List, Tuple, Dict


# ---------------------------------------------------------------------------
# Count tables
# ---------------------------------------------------------------------------

class NgramCounts:
    """Count tables for orders 1..max_order, built from one token stream.

    counts[n] is a dict: n-gram tuple (length n) -> count
    context_counts[n] is a dict: (n-1)-gram tuple -> total count of that
        context (i.e. sum of counts[n] over all n-grams sharing that prefix)
        -- stored separately to avoid recomputing sums repeatedly.
    """

    def __init__(self, stream: List[str], max_order: int = 4):
        self.max_order = max_order
        self.counts: Dict[int, Dict[Tuple[str, ...], int]] = {
            n: defaultdict(int) for n in range(1, max_order + 1)
        }
        self.vocab_tokens = set()

        for n in range(1, max_order + 1):
            for i in range(len(stream) - n + 1):
                ngram = tuple(stream[i:i + n])
                self.counts[n][ngram] += 1

        self.vocab_tokens = set(stream)
        self.vocab_size = len(self.vocab_tokens)

        # context_counts[n][context] = sum over all words w of counts[n][context + (w,)]
        # equivalently: context_counts[n] == counts[n-1] restricted to contexts
        # that actually appear as an (n-1)-gram -- but safer to just recompute
        # from counts[n] directly so it's always consistent with counts[n].
        self.context_counts: Dict[int, Dict[Tuple[str, ...], int]] = {
            n: defaultdict(int) for n in range(1, max_order + 1)
        }
        for n in range(1, max_order + 1):
            for ngram, c in self.counts[n].items():
                context = ngram[:-1]
                self.context_counts[n][context] += c

        # Count-of-counts tables per order, for Good-Turing: N_r = number of
        # distinct n-grams of this order that occurred exactly r times.
        self.count_of_counts: Dict[int, Counter] = {}
        for n in range(1, max_order + 1):
            self.count_of_counts[n] = Counter(self.counts[n].values())


# ---------------------------------------------------------------------------
# Add-k smoothing
# ---------------------------------------------------------------------------

class AddKModel:
    """Add-k (Laplace generalization) smoothing for a fixed n-gram order."""

    def __init__(self, ngram_counts: NgramCounts, order: int, k: float):
        self.nc = ngram_counts
        self.order = order
        self.k = k
        self.V = ngram_counts.vocab_size

    def logprob(self, context: Tuple[str, ...], word: str) -> float:
        context = context[-(self.order - 1):] if self.order > 1 else ()
        ngram = context + (word,)
        c_ngram = self.nc.counts[self.order].get(ngram, 0)
        c_context = self.nc.context_counts[self.order].get(context, 0)
        prob = (c_ngram + self.k) / (c_context + self.k * self.V)
        return math.log(prob)


# ---------------------------------------------------------------------------
# Good-Turing smoothing
# ---------------------------------------------------------------------------

class GoodTuringModel:
    """Good-Turing discounting for r <= gt_max; backoff to (order-1) MLE for
    unseen (r=0) n-grams (see PLAN.md assumption 3.3).

    Adjusted count: r* = (r+1) * N_{r+1} / N_r  for r in [1, gt_max]
    For r=0 (unseen n-gram): no direct GT estimate exists (N_0 is not a count
    we observe in the usual formula), so we fall back to a simple backoff:
    distribute a held-out mass (estimated as N_1 / N_total, i.e. the
    Good-Turing estimate for total unseen mass) proportionally via the
    (order-1)-gram MLE distribution. This is a standard simple Katz-style
    backoff, not full Katz backoff with its own discounting at every level --
    kept deliberately simple per the project's "minimal effort, ship it"
    principle, documented explicitly in the report.
    """

    def __init__(self, ngram_counts: NgramCounts, order: int, gt_max: int = 5):
        self.nc = ngram_counts
        self.order = order
        self.gt_max = gt_max
        self.V = ngram_counts.vocab_size

        coc = ngram_counts.count_of_counts[order]
        self.N_total = sum(r * n for r, n in coc.items())  # total token count at this order

        # Precompute adjusted counts r* for r = 1..gt_max
        self.r_star = {}
        for r in range(1, gt_max + 1):
            Nr = coc.get(r, 0)
            Nr1 = coc.get(r + 1, 0)
            if Nr == 0:
                self.r_star[r] = float(r)  # no data to adjust, fall back to raw count
            else:
                self.r_star[r] = (r + 1) * Nr1 / Nr if Nr1 > 0 else float(r)

        # Mass reserved for unseen n-grams (r=0): N_1 / N_total (standard GT)
        N1 = coc.get(1, 0)
        self.unseen_mass = (N1 / self.N_total) if self.N_total > 0 else 1e-10

        # Lower-order model for backing off unseen n-grams (order-1 MLE,
        # or uniform if order==1)
        if order > 1:
            self._lower = GoodTuringModel(ngram_counts, order - 1, gt_max) \
                if order - 1 >= 1 else None
        else:
            self._lower = None

    def _raw_count(self, context, word):
        ngram = context + (word,)
        return self.nc.counts[self.order].get(ngram, 0)

    def logprob(self, context: Tuple[str, ...], word: str) -> float:
        context = context[-(self.order - 1):] if self.order > 1 else ()
        r = self._raw_count(context, word)
        c_context = self.nc.context_counts[self.order].get(context, 0)

        if r == 0:
            # Backoff: unseen_mass spread over OOV-given-context, approximated
            # via lower-order distribution (or uniform at unigram level).
            if self.order == 1 or self._lower is None:
                prob = self.unseen_mass / max(self.V, 1)
            else:
                lower_context = context[1:] if len(context) > 0 else ()
                lower_logprob = self._lower.logprob(lower_context, word)
                prob = self.unseen_mass * math.exp(lower_logprob)
            prob = max(prob, 1e-12)  # floor to avoid log(0)
            return math.log(prob)
        elif r <= self.gt_max:
            r_adj = self.r_star[r]
            prob = r_adj / c_context if c_context > 0 else r_adj / max(self.V, 1)
            return math.log(max(prob, 1e-12))
        else:
            # r > gt_max: use raw MLE, undiscounted (standard GT practice --
            # only low counts are unreliable enough to need discounting)
            prob = r / c_context
            return math.log(prob)


# ---------------------------------------------------------------------------
# Interpolated Kneser-Ney (+ outer unigram blend, per PLAN.md 3.4)
# ---------------------------------------------------------------------------

class KneserNeyModel:
    """Standard recursive interpolated KN with fixed discount d, PLUS an
    outer global interpolation with the unigram distribution:

        P_final(w | context) = lam * P_KN(w | context) + (1-lam) * P_unigram(w)

    lam is tuned by the caller via grid search on validation perplexity (see
    `tune_kn_lambda` below) and passed in at construction time.

    See PLAN.md assumption 3.4 for why this two-part structure was chosen to
    resolve the spec's ambiguous "d=0.75 AND separately tune lambda" wording.
    """

    def __init__(self, ngram_counts: NgramCounts, order: int, d: float = 0.75,
                 lam: float = 1.0):
        self.nc = ngram_counts
        self.order = order
        self.d = d
        self.lam = lam
        self.V = ngram_counts.vocab_size

        # Precompute unigram MLE distribution for the outer blend.
        unigram_counts = ngram_counts.counts[1]
        total_unigrams = sum(unigram_counts.values())
        self._unigram_prob = {
            w: c / total_unigrams for (w,), c in unigram_counts.items()
        }
        self._unigram_floor = 1e-10  # for words with zero unigram count (shouldn't happen if w in vocab)

        # Precompute continuation counts for true KN lower-order smoothing:
        # continuation count of a word w at order n = number of distinct
        # (n)-gram contexts of order (n+1) in which w appears as the last
        # token. Needed for orders < max_order when used as the recursive
        # backoff base. We precompute this once per order pair actually used.
        self._continuation_cache = {}

        # PERFORMANCE: precompute, for every order from 1..self.order, the
        # number of distinct words following each context -- i.e.
        # distinct_following_table[n][context] = |{w : (context,w) in counts[n]}|.
        # Built once here (O(number of n-grams), not per logprob() call) so
        # `_distinct_following` below is an O(1) dict lookup instead of a full
        # linear scan over the n-gram table on every single token scored.
        # This matters a great deal at real corpus scale (~1.2M tokens):
        # without this, KN perplexity would be impractically slow.
        self._distinct_following_table: Dict[int, Dict[Tuple[str, ...], int]] = {}
        for n in range(1, self.order + 1):
            table = defaultdict(set)
            for ngram in ngram_counts.counts[n]:
                prefix, last = ngram[:-1], ngram[-1]
                table[prefix].add(last)
            self._distinct_following_table[n] = {k: len(v) for k, v in table.items()}

    def _continuation_prob(self, word: str, order: int) -> float:
        """P_continuation(word) at the given order: fraction of distinct
        (order+1)-gram types that end in `word`, used as the lowest-order KN
        base distribution (standard interpolated KN definition)."""
        cache_key = order
        if cache_key not in self._continuation_cache:
            higher = self.nc.counts.get(order + 1, {})
            # distinct contexts (order-grams) that precede each distinct
            # continuation word
            continuation_counts = defaultdict(set)
            for ngram in higher:
                prefix, last = ngram[:-1], ngram[-1]
                continuation_counts[last].add(prefix)
            total_types = sum(len(v) for v in continuation_counts.values())
            self._continuation_cache[cache_key] = (
                {w: len(ctxs) for w, ctxs in continuation_counts.items()},
                total_types,
            )
        type_counts, total_types = self._continuation_cache[cache_key]
        if total_types == 0:
            return 1.0 / max(self.V, 1)
        return type_counts.get(word, 0) / total_types

    def _kn_recursive(self, context: Tuple[str, ...], word: str, order: int) -> float:
        """Standard recursive interpolated KN probability (returns a plain
        probability, not log), for a given order, recursing down to the
        continuation-based unigram base case."""
        if order == 1:
            return self._continuation_prob(word, 1)

        ctx = context[-(order - 1):] if order > 1 else ()
        ngram = ctx + (word,)
        c_ngram = self.nc.counts[order].get(ngram, 0)
        c_context = self.nc.context_counts[order].get(ctx, 0)

        if c_context == 0:
            # Never saw this context at all -- fall back entirely to lower order
            return self._kn_recursive(context, word, order - 1)

        # Number of distinct word types following this context (for the
        # normalizing backoff weight lambda(context))
        # We need: distinct continuations of `ctx` at this order.
        distinct_continuations = self._distinct_following(ctx, order)

        discounted = max(c_ngram - self.d, 0.0) / c_context
        backoff_weight = (self.d * distinct_continuations) / c_context
        lower = self._kn_recursive(context, word, order - 1)

        return discounted + backoff_weight * lower

    def _distinct_following(self, context: Tuple[str, ...], order: int) -> int:
        """Number of distinct words w such that (context + (w,)) was seen at
        this order -- used for the KN backoff normalizing weight. O(1)
        lookup into the table precomputed in __init__ (see PERFORMANCE note
        there) -- do not revert this to a linear scan."""
        return self._distinct_following_table.get(order, {}).get(context, 0)

    def logprob(self, context: Tuple[str, ...], word: str) -> float:
        p_kn = self._kn_recursive(context, word, self.order)
        p_uni = self._unigram_prob.get(word, self._unigram_floor)
        prob = self.lam * p_kn + (1 - self.lam) * p_uni
        return math.log(max(prob, 1e-12))


# ---------------------------------------------------------------------------
# Perplexity evaluation (shared by all 5 variants)
# ---------------------------------------------------------------------------

def perplexity(model, stream: List[str], order: int) -> float:
    """Compute perplexity of `model` over `stream`, using sliding windows of
    the given order. `stream` should already include <s>/</s> boundaries
    (i.e. it's the output of tokenizer.build_scoring_stream).

    Convention (PLAN.md 3.2): every token is scored INCLUDING </s>, EXCLUDING
    <s> as a predicted token (it only ever appears as context).
    """
    from tokenizer import BOS

    total_logprob = 0.0
    n_tokens = 0

    for i in range(len(stream)):
        word = stream[i]
        if word == BOS:
            continue  # never score <s> itself, it's context-only
        context_start = max(0, i - (order - 1))
        context = tuple(stream[context_start:i])
        total_logprob += model.logprob(context, word)
        n_tokens += 1

    avg_neg_logprob = -total_logprob / n_tokens
    return math.exp(avg_neg_logprob)


# ---------------------------------------------------------------------------
# KN lambda tuning (grid search on validation, per PLAN.md 3.4)
# ---------------------------------------------------------------------------

def tune_kn_lambda(ngram_counts: NgramCounts, order: int, val_stream: List[str],
                    d: float = 0.75, grid=None) -> Tuple[float, float]:
    """Grid search lam over `grid` (default {0.1,...,0.9}), minimizing
    validation perplexity. Returns (best_lam, best_val_ppl)."""
    if grid is None:
        grid = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

    best_lam, best_ppl = None, float("inf")
    for lam in grid:
        model = KneserNeyModel(ngram_counts, order, d=d, lam=lam)
        ppl = perplexity(model, val_stream, order)
        if ppl < best_ppl:
            best_lam, best_ppl = lam, ppl
    return best_lam, best_ppl


if __name__ == "__main__":
    # Smoke test on fabricated data -- NOT real DailyDialog, just verifies
    # the module runs end-to-end and produces sane (finite, >1) perplexities
    # before we touch Colab.
    import sys
    sys.path.insert(0, "/home/claude")
    from tokenizer import build_scoring_stream, Vocab, build_vocab, tokenize, join_dialogue

    fake_dialogues = [
        ["I like cats.", "Cats are nice.", "Do you like dogs too?"],
        ["I like dogs.", "Dogs are loyal.", "Cats are independent."],
        ["Do you like cats or dogs?", "I like both animals."],
        ["Cats are independent.", "Dogs are loyal and nice."],
    ] * 5  # repeat for slightly less sparse counts in this tiny smoke test

    token_lists = [tokenize(join_dialogue(d)) for d in fake_dialogues]
    vocab = build_vocab(token_lists, max_size=1000, min_count=1)
    stream = build_scoring_stream(fake_dialogues, vocab)

    print("Stream length:", len(stream), "Vocab size:", len(vocab))

    nc = NgramCounts(stream, max_order=4)

    print("\n--- Add-k (k=1), trigram ---")
    addk = AddKModel(nc, order=3, k=1.0)
    print("PPL:", perplexity(addk, stream, order=3))

    print("\n--- Good-Turing, trigram ---")
    gt = GoodTuringModel(nc, order=3, gt_max=5)
    print("PPL:", perplexity(gt, stream, order=3))

    print("\n--- Kneser-Ney, trigram (lam=1.0, i.e. pure KN, no unigram blend) ---")
    kn = KneserNeyModel(nc, order=3, d=0.75, lam=1.0)
    print("PPL:", perplexity(kn, stream, order=3))

    print("\n--- KN lambda tuning (using the same stream as 'validation' -- smoke test only) ---")
    best_lam, best_ppl = tune_kn_lambda(nc, order=3, val_stream=stream)
    print("Best lambda:", best_lam, "Best PPL:", best_ppl)

    print("\nAll smoke tests ran without error.")

Overwriting ngram_lm.py


In [ ]:
%%writefile awd_lstm.py
"""
AWD-LSTM (Merity, Keskar, Socher 2017, arXiv:1708.02182) -- Task 2.

Written fresh against the paper's description rather than adapted from the
official repo (github.com/salesforce/awd-lstm-lm), which predates modern
PyTorch and uses APIs (e.g. old-style weight-norm hooks, Python 2 idioms)
that don't run cleanly on current torch versions. See PLAN.md assumption 3.6.

Components implemented (per spec step 2):
  - WeightDropLSTM: DropConnect (p=0.5 default) applied to the
    hidden-to-hidden weight matrix of each LSTM layer, reapplied (a fresh
    dropout mask) on every forward call during training.
  - Embedding dropout (p=0.1): drops entire embedding ROWS (whole word
    types), not individual activations -- this is what the paper calls
    "embedding dropout" (distinct from standard dropout on embedding
    outputs).
  - Output (locked) dropout (p=0.4): standard dropout applied to the final
    LSTM layer's output before the softmax, using the SAME mask across all
    timesteps in a sequence (a "locked"/"variational" dropout mask) --
    this is what the paper calls output dropout.
  - Tied embedding/softmax weights.

NOT implemented (explicit, documented deviations -- state in report):
  - AR/TAR activation regularization (the paper's alpha/beta regularization
    terms) -- spec's step-by-step instructions (steps 1-4) do not mention
    these, only DropConnect + embedding dropout + output dropout + NT-ASGD +
    the listed hyperparameters, so we implement exactly what's listed and
    flag AR/TAR as an omitted paper detail, consistent with "minimal effort,
    ship what's asked for" per team decision.
  - Variable-length BPTT (paper varies BPTT length per batch for
    regularization) -- spec says "BPTT length 70" as a fixed hyperparameter,
    so we use a fixed length, not the paper's variable scheme.
  - The paper's fine-tuning second stage -- out of scope per spec's step 5.
"""

import math

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    _TORCH_AVAILABLE = True
except ImportError:
    # This sandbox has no internet access to PyTorch's package servers, so
    # this module cannot be executed/tested here -- only written and
    # manually shape-traced (see SHAPE_TRACE below). It WILL run in Colab,
    # which has torch preinstalled. Running this file directly in an
    # environment without torch still prints the shape trace for review.
    _TORCH_AVAILABLE = False
    nn = None
    F = None


# ---------------------------------------------------------------------------
# DropConnect on hidden-to-hidden weights
# ---------------------------------------------------------------------------

class WeightDropLSTM(nn.Module if _TORCH_AVAILABLE else object):
    """Wraps nn.LSTM and applies DropConnect to the hidden-to-hidden weight
    matrix (weight_hh_l0) on every forward call during training.

    Standard dropout zeroes ACTIVATIONS; DropConnect zeroes WEIGHTS.

    IMPLEMENTATION NOTE (important, read before modifying): an earlier draft
    of this class tried to replace nn.LSTM's weight_hh_l0 Parameter with a
    plain (non-leaf) tensor via setattr(), mirroring a trick from the
    official AWD-LSTM repo. That approach is fragile on modern PyTorch:
    nn.LSTM caches parameter names/references at construction time
    (`_flat_weights`, `_all_weights`) for its fused cuDNN/CPU kernel path,
    and a bare setattr() does not reliably update what the kernel actually
    reads -- this could silently train with the UNDROPPED weight while
    appearing to work. Since this module cannot be executed/tested in this
    sandbox (no torch available -- see top-of-file note) before being run
    for real in Colab, we deliberately avoid that risk entirely and use a
    simpler, more robust approach instead:

      1. Keep weight_hh_l0 as nn.LSTM's own real Parameter (untouched,
         nothing deleted/reassigned/re-registered).
      2. On each forward() call during training: clone the current
         weight_hh_l0.data, compute a dropped-out version, write it IN
         PLACE into weight_hh_l0.data (same tensor object, so cuDNN's
         cached references stay valid), run the forward pass, then
         restore the original (undropped) data afterward.
      3. Gradients still flow correctly: autograd records the forward
         computation using whatever data was in weight_hh_l0 at the time
         (the dropped version), and since the Parameter object itself
         (and its .grad) is never replaced, the optimizer's step() still
         updates the SAME Parameter it always did -- we're only ever
         swapping the underlying .data buffer's contents, never the
         Parameter object's identity.

    This trades a small amount of complexity (explicit save/restore) for
    something we can reason about correctly without being able to run it
    first. If Colab shows the model training (loss decreasing, no shape
    errors) in the smoke test, this approach is confirmed working.
    """

    def __init__(self, input_size, hidden_size, dropconnect_p=0.5):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=1, batch_first=False)
        self.dropconnect_p = dropconnect_p

    def forward(self, x, hidden=None):
        if self.training and self.dropconnect_p > 0:
            original_data = self.lstm.weight_hh_l0.data.clone()
            mask = torch.bernoulli(
                original_data.new_full(original_data.shape, 1 - self.dropconnect_p)
            ) / (1 - self.dropconnect_p)  # inverted dropout scaling
            self.lstm.weight_hh_l0.data.mul_(mask)  # in-place: same tensor object
            try:
                output, new_hidden = self.lstm(x, hidden)
            finally:
                # Always restore, even if the forward pass raises, so a
                # mid-training exception can't leave the model's real
                # weights permanently corrupted.
                self.lstm.weight_hh_l0.data.copy_(original_data)
            return output, new_hidden
        else:
            return self.lstm(x, hidden)


# ---------------------------------------------------------------------------
# Embedding dropout (drops whole word-rows, not individual activations)
# ---------------------------------------------------------------------------

def embedded_dropout(embed_weight, input_ids, p=0.1, training=True):
    """Apply embedding dropout: with probability p, zero out an ENTIRE row
    of the embedding matrix (i.e. drop a word TYPE for this forward pass,
    so every occurrence of that word in the batch gets a zero vector), then
    do a normal embedding lookup with the (possibly zeroed) matrix.
    """
    if training and p > 0:
        mask = embed_weight.data.new_full((embed_weight.size(0), 1), 1 - p)
        mask = torch.bernoulli(mask) / (1 - p)
        masked_weight = embed_weight * mask
    else:
        masked_weight = embed_weight
    return F.embedding(input_ids, masked_weight)


# ---------------------------------------------------------------------------
# Locked (variational) dropout -- same mask reused across all timesteps
# ---------------------------------------------------------------------------

class LockedDropout(nn.Module if _TORCH_AVAILABLE else object):
    """Standard dropout would sample a new mask per timestep; locked dropout
    samples ONE mask and reuses it for every timestep in the sequence
    (standard technique from the AWD-LSTM paper, used here for output
    dropout, p=0.4)."""

    def forward(self, x, p):
        if not self.training or p == 0:
            return x
        # x shape: (seq_len, batch, hidden)
        mask = x.data.new_full((1, x.size(1), x.size(2)), 1 - p)
        mask = torch.bernoulli(mask) / (1 - p)
        mask = mask.expand_as(x)
        return x * mask


# ---------------------------------------------------------------------------
# Full model
# ---------------------------------------------------------------------------

class AWDLSTM(nn.Module if _TORCH_AVAILABLE else object):
    """3-layer weight-dropped LSTM LM with tied embeddings.

    Args match the spec exactly:
        vocab_size: size of the vocabulary (len(vocab) from tokenizer.Vocab)
        embed_dim: 400 (paper default)
        hidden_dim: 1150 (paper default; pass 400 for the compute-constrained
            fallback -- see PLAN.md assumption 3.6)
        n_layers: 3
        dropconnect_p: 0.5 (hidden-to-hidden DropConnect)
        embed_dropout_p: 0.1
        output_dropout_p: 0.4

    IMPORTANT architectural note on tied weights: tying requires the LAST
    LSTM layer's hidden_dim to equal embed_dim (since the output projection
    reuses the embedding matrix, shape (vocab_size, embed_dim)). The paper
    handles this by making the LAST layer's hidden size equal to embed_dim
    (400) even though earlier layers use the larger hidden_dim (1150). We
    replicate that: layers 0..n_layers-2 use hidden_dim, the FINAL layer
    uses embed_dim as its output size.
    """

    def __init__(self, vocab_size, embed_dim=400, hidden_dim=1150, n_layers=3,
                 dropconnect_p=0.5, embed_dropout_p=0.1, output_dropout_p=0.4):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embed_dropout_p = embed_dropout_p
        self.output_dropout_p = output_dropout_p

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # Per-layer sizes: all but last use hidden_dim, last uses embed_dim
        # (required for weight tying -- see docstring).
        layer_input_sizes = [embed_dim] + [hidden_dim] * (n_layers - 1)
        layer_output_sizes = [hidden_dim] * (n_layers - 1) + [embed_dim]

        self.rnns = nn.ModuleList([
            WeightDropLSTM(layer_input_sizes[i], layer_output_sizes[i], dropconnect_p)
            for i in range(n_layers)
        ])

        self.locked_dropout = LockedDropout()

        self.decoder = nn.Linear(embed_dim, vocab_size)
        # Tie weights: decoder.weight (vocab_size, embed_dim) == embedding.weight
        self.decoder.weight = self.embedding.weight

        self._init_weights()

    def _init_weights(self):
        init_range = 0.1
        self.embedding.weight.data.uniform_(-init_range, init_range)
        self.decoder.bias.data.zero_()
        # decoder.weight is tied to embedding.weight, already initialized above

    def init_hidden(self, batch_size, device):
        hidden = []
        for i, rnn in enumerate(self.rnns):
            h_size = self.hidden_dim if i < self.n_layers - 1 else self.embed_dim
            h = torch.zeros(1, batch_size, h_size, device=device)
            c = torch.zeros(1, batch_size, h_size, device=device)
            hidden.append((h, c))
        return hidden

    def forward(self, input_ids, hidden):
        """
        input_ids: (seq_len, batch) LongTensor of token ids
        hidden: list of (h, c) tuples, one per layer, from init_hidden() or
                a previous forward() call's returned hidden state (for
                truncated BPTT -- caller detaches between chunks)

        Returns:
            logits: (seq_len, batch, vocab_size)
            new_hidden: list of (h, c) tuples, same structure as `hidden`
        """
        emb = embedded_dropout(
            self.embedding.weight, input_ids,
            p=self.embed_dropout_p, training=self.training
        )  # (seq_len, batch, embed_dim)

        x = emb
        new_hidden = []
        for i, rnn in enumerate(self.rnns):
            x, h_new = rnn(x, hidden[i])
            new_hidden.append(h_new)
            if i < self.n_layers - 1:
                # Locked dropout between LSTM layers too (standard AWD-LSTM
                # practice uses this for inter-layer dropout as well, at the
                # same output_dropout_p -- spec doesn't separately specify an
                # inter-layer rate, so we reuse output_dropout_p here, a
                # reasonable simplification per "minimal effort" principle).
                x = self.locked_dropout(x, self.output_dropout_p)

        x = self.locked_dropout(x, self.output_dropout_p)  # final output dropout
        logits = self.decoder(x)  # (seq_len, batch, vocab_size)
        return logits, new_hidden

    @staticmethod
    def detach_hidden(hidden):
        """Detach hidden state from the computation graph between BPTT
        chunks (standard truncated-BPTT practice -- prevents backprop from
        running all the way back to the start of training)."""
        return [(h.detach(), c.detach()) for (h, c) in hidden]


# ---------------------------------------------------------------------------
# Manual shape trace (since we cannot run torch in this sandbox -- see module
# docstring). This documents, step by step, what every tensor shape SHOULD
# be, so the first real Colab run can be checked against this by eye instead
# of debugging blind.
# ---------------------------------------------------------------------------

SHAPE_TRACE = """
Manual shape trace for AWDLSTM(vocab_size=12482, embed_dim=400, hidden_dim=1150, n_layers=3)
with seq_len=70 (BPTT length per spec), batch_size=80 (per spec):

  input_ids:            (70, 80)                      int64
  embedding lookup ->   (70, 80, 400)                  float32
  layer 0 (WeightDropLSTM, in=400, out=1150):
    input  (70, 80, 400)
    output (70, 80, 1150), h/c each (1, 80, 1150)
  locked_dropout (p=0.4) -> same shape (70, 80, 1150)
  layer 1 (WeightDropLSTM, in=1150, out=1150):
    input  (70, 80, 1150)
    output (70, 80, 1150), h/c each (1, 80, 1150)
  locked_dropout (p=0.4) -> same shape (70, 80, 1150)
  layer 2 (WeightDropLSTM, in=1150, out=400):   <- last layer outputs embed_dim, for tying
    input  (70, 80, 1150)
    output (70, 80, 400), h/c each (1, 80, 400)
  locked_dropout (p=0.4, final output dropout) -> (70, 80, 400)
  decoder (Linear(400 -> 12482), weight tied to embedding.weight):
    output logits: (70, 80, 12482)

Parameter count estimate (rough, for sanity-checking against Colab's actual
printed count -- see the test script's parameter counter):
  embedding: 12482 * 400                           = 4,992,800
  layer 0 LSTM: 4 * 1150 * (400+1150+1)            = 7,135,000   (approx, incl. biases)
  layer 1 LSTM: 4 * 1150 * (1150+1150+1)           = 10,580,600  (approx)
  layer 2 LSTM: 4 * 400  * (1150+400+1)            = 2,482,400   (approx)
  decoder bias: 12482
  decoder.weight is TIED (0 additional params, reuses embedding)
  ------------------------------------------------------------------
  TOTAL approx: ~25.2M parameters

This is in the same ballpark as the paper's ~24M for PTB/WT2 configs, which
is expected since embed_dim/hidden_dim match the paper despite our much
smaller vocab (12,482 vs ~33K) -- the LSTM weight matrices dominate
parameter count, not the embedding/decoder, so vocab size differences barely
move the total.
"""

# ---------------------------------------------------------------------------
# Training / evaluation utility functions
#
# These are plain tensor-reshaping and bookkeeping helpers -- they do NOT
# touch the WeightDropLSTM internals, so they carry none of the non-leaf-
# tensor risk that the weight-drop trick does. Kept deliberately simple.
# ---------------------------------------------------------------------------

def repackage_hidden(h):
    """Detach hidden states from their computation-graph history, so
    truncated BPTT doesn't backprop through the entire training history.
    Call this once per batch, on the hidden state returned by the previous
    batch's forward() call, before feeding it into the next batch."""
    if _TORCH_AVAILABLE and isinstance(h, torch.Tensor):
        return h.detach()
    else:
        return tuple(repackage_hidden(v) for v in h)


def batchify(data_ids, bsz, device):
    """Reshape a flat list[int] of token ids into a (num_steps, bsz) tensor
    for truncated-BPTT training, per spec: batch size 80.

    Standard LM batching: split the flat stream into `bsz` contiguous
    columns (dropping leftover tokens that don't fill a full column), so
    each column is read top-to-bottom as one continuous sub-sequence and
    hidden state carries across batches WITHIN a column via repackage_hidden.
    """
    data = torch.tensor(data_ids, dtype=torch.long)
    nbatch = data.size(0) // bsz
    data = data.narrow(0, 0, nbatch * bsz)
    data = data.view(bsz, -1).t().contiguous()
    return data.to(device)


def get_batch(source, i, bptt=70):
    """Slice one training chunk starting at row i: input is source[i:i+seq_len],
    target is the same window shifted forward by one timestep (next-token
    prediction), flattened to 1D for CrossEntropyLoss. seq_len is clipped
    near the end of `source` so we never index out of bounds."""
    seq_len = min(bptt, len(source) - 1 - i)
    data = source[i:i + seq_len]
    target = source[i + 1:i + 1 + seq_len].reshape(-1)
    return data, target


@torch.no_grad() if _TORCH_AVAILABLE else (lambda f: f)
def evaluate_stream_exact(model, stream_ids, bos_id, device, chunk_size=2048,
                           return_token_logprobs=False):
    """Exact sequential perplexity evaluation over a 1D scoring stream
    (batch_size=1, no truncation/dropped tail), matching Task 1's n-gram
    perplexity convention exactly (PLAN.md 3.2/3.7): every token is scored
    EXCEPT <s> (bos_id), which only ever appears as context, never as a
    predicted target. This is what makes Task 1 and Task 2 perplexities
    directly comparable, and is REQUIRED for Task 3's per-position hybrid
    interpolation to be well-defined (both models must score the identical
    sequence of positions).

    Use this for all REPORTED perplexity numbers (val/test). Use the faster
    batched `batchify`+`get_batch` path only for TRAINING, where exact
    per-token alignment with Task 1 doesn't matter (training just needs the
    loss signal, not a comparable metric).
    """
    model.eval()
    data = torch.tensor(stream_ids, dtype=torch.long, device=device).unsqueeze(1)  # (N, 1)
    hidden = model.init_hidden(1, device)

    total_logprob = 0.0
    n_tokens = 0
    token_logprobs = []

    for i in range(0, data.size(0) - 1, chunk_size):
        seq_len = min(chunk_size, data.size(0) - 1 - i)
        inp = data[i:i + seq_len]                      # (seq_len, 1)
        tgt = data[i + 1:i + 1 + seq_len].squeeze(1)    # (seq_len,)

        logits, hidden = model(inp, hidden)
        hidden = repackage_hidden(hidden)

        log_probs = F.log_softmax(logits.squeeze(1), dim=-1)  # (seq_len, V)
        target_lps = log_probs.gather(1, tgt.unsqueeze(1)).squeeze(1)  # (seq_len,)

        valid_mask = (tgt != bos_id)
        valid_lps = target_lps[valid_mask]

        total_logprob += valid_lps.sum().item()
        n_tokens += valid_mask.sum().item()

        if return_token_logprobs:
            token_logprobs.extend(valid_lps.cpu().tolist())

    ppl = math.exp(-total_logprob / n_tokens)
    if return_token_logprobs:
        return ppl, token_logprobs
    return ppl


if __name__ == "__main__":
    print(SHAPE_TRACE)
    print("\n(Full forward-pass execution requires torch + will be run in Colab;")
    print(" this file was written and shape-traced by hand in this sandbox")
    print(" since torch is not installable here -- verify against real run.)")

Overwriting awd_lstm.py


In [ ]:
%%writefile hybrid.py
"""
Task 3: linear interpolation of Task 1's 4-gram KN model and Task 2's
AWD-LSTM, per PLAN.md Section 5 (Task 3).

P_hybrid(w_i | context) = lambda * P_KN(w_i | context) + (1-lambda) * P_AWD(w_i | context)

Core requirement (PLAN.md 3.7): both models must be scored over the EXACT
SAME sequence of positions for this interpolation to be well-defined. We
reuse the same scoring-stream convention from tokenizer.build_scoring_stream
that both Task 1 and Task 2 already evaluate perplexity with, so this is
already guaranteed by construction as long as both models are called on the
same `stream` list.

Design: extract per-token log-probs from EACH model SEPARATELY into plain
Python lists of floats (same length, same order, position i in one list
corresponds to the same token as position i in the other). Then interpolate
in plain Python/numpy -- simple, easy to verify, and fast enough at this
corpus scale (test stream is ~108K tokens).
"""

import math
from typing import List

try:
    import torch
    import torch.nn.functional as F
    _TORCH_AVAILABLE = True
except ImportError:
    _TORCH_AVAILABLE = False


# ---------------------------------------------------------------------------
# Per-token log-prob extraction -- n-gram side
# ---------------------------------------------------------------------------

def ngram_token_logprobs(model, stream: List[str], order: int) -> List[float]:
    """Returns one log-prob per SCORED token in `stream` (i.e. every token
    except <s>, matching the convention in ngram_lm.perplexity /
    tokenizer.build_scoring_stream). Length == number of non-<s> tokens in
    `stream`.
    """
    from tokenizer import BOS

    logprobs = []
    for i in range(len(stream)):
        word = stream[i]
        if word == BOS:
            continue
        context_start = max(0, i - (order - 1))
        context = tuple(stream[context_start:i])
        logprobs.append(model.logprob(context, word))
    return logprobs


# ---------------------------------------------------------------------------
# Per-token log-prob extraction -- LSTM side
# ---------------------------------------------------------------------------

def lstm_token_logprobs(model, stream: List[str], vocab, bos_id: int, device,
                         chunk_size: int = 2048) -> List[float]:
    """Returns one log-prob per SCORED token in `stream` (every token except
    <s>), in the SAME order as ngram_token_logprobs would produce for the
    same stream -- i.e. skip the log-prob that would have been assigned to
    predicting a <s> token (since <s> is never a prediction target, only
    ever appears as context), exactly mirroring ngram_lm.perplexity's and
    awd_lstm.evaluate_stream_exact's shared convention.

    This is almost identical to awd_lstm.evaluate_stream_exact, except it
    RETURNS the full list of per-token log-probs instead of collapsing them
    into a single perplexity number -- we need the individual values here to
    interpolate position-by-position with the n-gram model's values.
    """
    ids = vocab.encode(stream)
    data = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(1)  # (N, 1)
    hidden = model.init_hidden(1, device)

    logprobs = []
    model.eval()
    with torch.no_grad():
        for i in range(0, data.size(0) - 1, chunk_size):
            seq_len = min(chunk_size, data.size(0) - 1 - i)
            inp = data[i:i + seq_len]
            tgt = data[i + 1:i + 1 + seq_len].squeeze(1)

            logits, hidden = model(inp, hidden)
            from awd_lstm import repackage_hidden
            hidden = repackage_hidden(hidden)

            log_probs = F.log_softmax(logits.squeeze(1), dim=-1)
            target_lps = log_probs.gather(1, tgt.unsqueeze(1)).squeeze(1)

            valid_mask = (tgt != bos_id)
            valid_lps = target_lps[valid_mask]
            logprobs.extend(valid_lps.cpu().tolist())

    return logprobs


# ---------------------------------------------------------------------------
# Interpolation + perplexity from precomputed per-token log-probs
# ---------------------------------------------------------------------------

def interpolate_perplexity(kn_logprobs: List[float], lstm_logprobs: List[float],
                            lam: float) -> float:
    """Given two aligned lists of per-token NATURAL-LOG probabilities (same
    length, position i = same token in both), compute the perplexity of the
    linearly-interpolated hybrid:

        P_hybrid(token_i) = lam * exp(kn_logprobs[i]) + (1-lam) * exp(lstm_logprobs[i])

    Interpolation happens in PROBABILITY space (not log space) -- this is
    standard for linear interpolation of LMs, since the whole point is a
    weighted average of probabilities, not of log-probabilities (which
    would be a geometric-mean-like blend instead, a different and NOT what
    the spec's formula P_hybrid = lambda*P_KN + (1-lambda)*P_AWD specifies).
    """
    assert len(kn_logprobs) == len(lstm_logprobs), (
        f"Misaligned token streams: KN has {len(kn_logprobs)} scored tokens, "
        f"LSTM has {len(lstm_logprobs)} -- these MUST match exactly for "
        f"interpolation to be valid. Check that both were computed from the "
        f"identical `stream` list."
    )

    total_logprob = 0.0
    n = len(kn_logprobs)
    for kn_lp, lstm_lp in zip(kn_logprobs, lstm_logprobs):
        p_kn = math.exp(kn_lp)
        p_lstm = math.exp(lstm_lp)
        p_hybrid = lam * p_kn + (1 - lam) * p_lstm
        p_hybrid = max(p_hybrid, 1e-12)  # floor to avoid log(0)
        total_logprob += math.log(p_hybrid)

    return math.exp(-total_logprob / n)


def tune_hybrid_lambda(kn_val_logprobs: List[float], lstm_val_logprobs: List[float],
                        grid=None):
    """Grid search lambda in [0.0, 1.0] (step 0.1 by default, per spec),
    minimizing validation perplexity. Returns (best_lambda, best_val_ppl,
    all_results) where all_results is a list of (lambda, val_ppl) pairs for
    plotting the full curve.
    """
    if grid is None:
        grid = [round(x * 0.1, 1) for x in range(0, 11)]  # 0.0, 0.1, ..., 1.0

    results = []
    best_lam, best_ppl = None, float("inf")
    for lam in grid:
        ppl = interpolate_perplexity(kn_val_logprobs, lstm_val_logprobs, lam)
        results.append((lam, ppl))
        if ppl < best_ppl:
            best_lam, best_ppl = lam, ppl
    return best_lam, best_ppl, results


if __name__ == "__main__":
    # Smoke test with fabricated aligned log-prob lists (no torch/real models
    # needed for this part -- pure arithmetic verification).
    import random
    random.seed(42)

    # Simulate: KN is decent, LSTM is better, on a small fake "stream"
    n_tokens = 200
    kn_lps = [-random.uniform(2.0, 5.0) for _ in range(n_tokens)]
    lstm_lps = [-random.uniform(1.0, 4.0) for _ in range(n_tokens)]  # systematically better

    print("KN-only perplexity (lambda=1.0):", interpolate_perplexity(kn_lps, lstm_lps, 1.0))
    print("LSTM-only perplexity (lambda=0.0):", interpolate_perplexity(kn_lps, lstm_lps, 0.0))

    best_lam, best_ppl, all_results = tune_hybrid_lambda(kn_lps, lstm_lps)
    print("\nFull lambda sweep:")
    for lam, ppl in all_results:
        print(f"  lambda={lam}: PPL={ppl:.3f}")
    print(f"\nBest lambda: {best_lam}, Best PPL: {best_ppl:.3f}")

    # Sanity check: lambda=1.0 result should EXACTLY match a pure KN-only
    # perplexity computed directly (not through the hybrid formula), since
    # at lambda=1.0 the LSTM term is multiplied by (1-1.0)=0.
    direct_kn_only = math.exp(-sum(kn_lps) / len(kn_lps))
    hybrid_at_1 = interpolate_perplexity(kn_lps, lstm_lps, 1.0)
    print(f"\nSanity check -- direct KN-only PPL: {direct_kn_only:.6f}, "
          f"hybrid(lambda=1.0) PPL: {hybrid_at_1:.6f}, "
          f"match: {abs(direct_kn_only - hybrid_at_1) < 1e-6}")

    # Test misalignment detection
    try:
        interpolate_perplexity(kn_lps, lstm_lps[:-1], 0.5)
        print("ERROR: misalignment was NOT detected!")
    except AssertionError as e:
        print("\nMisalignment correctly detected:", str(e)[:80], "...")

Overwriting hybrid.py


In [ ]:
import importlib
import tokenizer as tok
import ngram_lm as ngl
import awd_lstm as awd
import hybrid
importlib.reload(tok)
importlib.reload(ngl)
importlib.reload(awd)
importlib.reload(hybrid)

import torch
import torch.nn.functional as F
import math, time, json, urllib.request
import pandas as pd

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.11.0+cu130
CUDA available: True


In [ ]:
urls = {
    "train": "https://huggingface.co/datasets/daily_dialog/resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet",
    "validation": "https://huggingface.co/datasets/daily_dialog/resolve/refs%2Fconvert%2Fparquet/default/validation/0000.parquet",
    "test": "https://huggingface.co/datasets/daily_dialog/resolve/refs%2Fconvert%2Fparquet/default/test/0000.parquet",
}
local_paths = {}
for split, url in urls.items():
    local_path = f"/content/daily_dialog_{split}.parquet"
    urllib.request.urlretrieve(url, local_path)
    local_paths[split] = local_path
dfs = {split: pd.read_parquet(path) for split, path in local_paths.items()}

train_token_lists = [tok.tokenize(tok.join_dialogue(d)) for d in dfs['train']['dialog'].tolist()]
vocab = tok.build_vocab(train_token_lists, max_size=30000, min_count=2)
print("Vocab size:", len(vocab))

train_stream = tok.build_scoring_stream(dfs['train']['dialog'].tolist(), vocab)
val_stream = tok.build_scoring_stream(dfs['validation']['dialog'].tolist(), vocab)
test_stream = tok.build_scoring_stream(dfs['test']['dialog'].tolist(), vocab)
print("Stream lengths:", len(train_stream), len(val_stream), len(test_stream))

t0 = time.time()
nc = ngl.NgramCounts(train_stream, max_order=4)
print("NgramCounts built in", time.time() - t0, "s")

kn_model = ngl.KneserNeyModel(nc, order=4, d=0.75, lam=0.9)
print("KN model ready")

Vocab size: 12482
Stream lengths: 1201536 110872 108079
NgramCounts built in 3.780426263809204 s
KN model ready


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CHECKPOINT_URL = "https://github.com/aryan-256/kn-lstm-hybrid/releases/download/v1.0-checkpoints/awd_lstm_final_averaged.pt"
checkpoint_path = "/content/awd_lstm_final_averaged.pt"

print("Downloading trained AWD-LSTM checkpoint...")
urllib.request.urlretrieve(CHECKPOINT_URL, checkpoint_path)
print("Downloaded.")

lstm_model = awd.AWDLSTM(
    vocab_size=len(vocab), embed_dim=400, hidden_dim=1150, n_layers=3,
    dropconnect_p=0.5, embed_dropout_p=0.1, output_dropout_p=0.4,
).to(device)
lstm_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
lstm_model.eval()

print("LSTM loaded. Parameter count:", sum(p.numel() for p in lstm_model.parameters()))

Downloaded.
LSTM loaded. Parameter count: 25216882


In [ ]:
bos_id = vocab.token2id[tok.BOS]

print("Extracting KN log-probs...")
kn_val_lps = hybrid.ngram_token_logprobs(kn_model, val_stream, order=4)
kn_test_lps = hybrid.ngram_token_logprobs(kn_model, test_stream, order=4)

print("Extracting LSTM log-probs...")
lstm_val_lps = hybrid.lstm_token_logprobs(lstm_model, val_stream, vocab, bos_id, device)
lstm_test_lps = hybrid.lstm_token_logprobs(lstm_model, test_stream, vocab, bos_id, device)

print("Alignment check:", len(kn_val_lps) == len(lstm_val_lps), len(kn_test_lps) == len(lstm_test_lps))

best_lam, best_val_ppl, lambda_sweep_results = hybrid.tune_hybrid_lambda(kn_val_lps, lstm_val_lps)
print(f"\nBest lambda: {best_lam}, Best validation PPL: {best_val_ppl:.3f}")
for lam, ppl in lambda_sweep_results:
    print(f"  lambda={lam}: val PPL={ppl:.3f}")

hybrid_test_ppl = hybrid.interpolate_perplexity(kn_test_lps, lstm_test_lps, best_lam)
print(f"\nHybrid test PPL (at lambda={best_lam}): {hybrid_test_ppl:.3f}")

Extracting KN log-probs...
Extracting LSTM log-probs...
Alignment check: True True

Best lambda: 0.4, Best validation PPL: 22.807
  lambda=0.0: val PPL=30.050
  lambda=0.1: val PPL=24.582
  lambda=0.2: val PPL=23.393
  lambda=0.3: val PPL=22.906
  lambda=0.4: val PPL=22.807
  lambda=0.5: val PPL=23.009
  lambda=0.6: val PPL=23.503
  lambda=0.7: val PPL=24.347
  lambda=0.8: val PPL=25.694
  lambda=0.9: val PPL=27.987
  lambda=1.0: val PPL=34.447

Hybrid test PPL (at lambda=0.4): 23.086


In [ ]:
BLIMP_PARADIGMS = [
    "anaphor_gender_agreement", "determiner_noun_agreement_1",
    "regular_plural_subject_verb_agreement_1", "wh_island", "npi_present_1",
    "left_branch_island_simple_question", "principle_A_domain_1",
    "drop_argument", "existential_there_quantifiers_1",
]

def load_blimp_paradigm(paradigm_name):
    url = f"https://huggingface.co/datasets/nyu-mll/blimp/resolve/main/{paradigm_name}/train-00000-of-00001.parquet"
    local_path = f"/content/blimp_{paradigm_name}.parquet"
    urllib.request.urlretrieve(url, local_path)
    return pd.read_parquet(local_path)

blimp_dfs = {p: load_blimp_paradigm(p) for p in BLIMP_PARADIGMS}
print("Loaded", len(blimp_dfs), "paradigms")

def score_kn(sentence, order=4):
    toks = tok.tokenize(sentence)
    toks = vocab.encode_str(toks)
    seq = [tok.BOS] + toks + [tok.EOS]
    lps = []
    for i in range(len(seq)):
        if seq[i] == tok.BOS:
            continue
        context_start = max(0, i - (order - 1))
        context = tuple(seq[context_start:i])
        lps.append(kn_model.logprob(context, seq[i]))
    return lps

def score_lstm(sentence):
    toks = tok.tokenize(sentence)
    toks = vocab.encode_str(toks)
    seq_str = [tok.BOS] + toks + [tok.EOS]
    ids = vocab.encode(seq_str)
    input_tensor = torch.tensor(ids[:-1], dtype=torch.long, device=device).unsqueeze(1)
    target_tensor = torch.tensor(ids[1:], dtype=torch.long, device=device)
    hidden = lstm_model.init_hidden(1, device)
    with torch.no_grad():
        logits, _ = lstm_model(input_tensor, hidden)
        log_probs = F.log_softmax(logits.squeeze(1), dim=-1)
        lps = log_probs.gather(1, target_tensor.unsqueeze(1)).squeeze(1).cpu().tolist()
    return lps

def score_sentence_hybrid(sentence, lam):
    kn_lps = score_kn(sentence)
    lstm_lps = score_lstm(sentence)
    assert len(kn_lps) == len(lstm_lps)
    total = 0.0
    for kn_lp, lstm_lp in zip(kn_lps, lstm_lps):
        p_hybrid = lam * math.exp(kn_lp) + (1 - lam) * math.exp(lstm_lp)
        total += math.log(max(p_hybrid, 1e-12))
    return total / len(kn_lps)

def evaluate_blimp_hybrid(lam, paradigms_dict):
    per_paradigm = {}
    total_correct, total_pairs = 0, 0
    for paradigm_name, df in paradigms_dict.items():
        correct = 0
        for _, row in df.iterrows():
            lp_good = score_sentence_hybrid(row['sentence_good'], lam)
            lp_bad = score_sentence_hybrid(row['sentence_bad'], lam)
            if lp_good > lp_bad:
                correct += 1
        acc = correct / len(df)
        per_paradigm[paradigm_name] = acc
        total_correct += correct
        total_pairs += len(df)
        print(f"{paradigm_name}: {acc:.3f} ({correct}/{len(df)})")
    overall_acc = total_correct / total_pairs
    print(f"\nOverall BLiMP accuracy (Hybrid): {overall_acc:.3f} ({total_correct}/{total_pairs})")
    return per_paradigm, overall_acc

per_paradigm_acc, overall_acc = evaluate_blimp_hybrid(best_lam, blimp_dfs)

Loaded 9 paradigms
anaphor_gender_agreement: 0.459 (459/1000)
determiner_noun_agreement_1: 0.642 (642/1000)
regular_plural_subject_verb_agreement_1: 0.590 (590/1000)
wh_island: 0.497 (497/1000)
npi_present_1: 0.470 (470/1000)
left_branch_island_simple_question: 0.693 (693/1000)
principle_A_domain_1: 0.999 (999/1000)
drop_argument: 0.485 (485/1000)
existential_there_quantifiers_1: 0.986 (986/1000)

Overall BLiMP accuracy (Hybrid): 0.647 (5821/9000)


In [14]:
print("=== TASK 3 FINAL SUMMARY ===")
print(f"Best lambda: {best_lam}")
print(f"Hybrid test perplexity: {hybrid_test_ppl:.3f}")
print(f"Hybrid BLiMP accuracy: {overall_acc:.3f}")
print("\nComparison:")
print("Task 1 (4-gram KN):  test PPL 33.06, BLiMP 56.1%")
print("Task 2 (AWD-LSTM):   test PPL 32.85, BLiMP 68.4%")
print(f"Task 3 (Hybrid):     test PPL {hybrid_test_ppl:.3f}, BLiMP {overall_acc:.3f}")
print("\nThis notebook required no retraining -- the AWD-LSTM checkpoint was")
print("downloaded directly from this repository's GitHub Release.")

=== TASK 3 FINAL SUMMARY ===
Best lambda: 0.4
Hybrid test perplexity: 23.086
Hybrid BLiMP accuracy: 0.647

Comparison:
Task 1 (4-gram KN):  test PPL 33.06, BLiMP 56.1%
Task 2 (AWD-LSTM):   test PPL 32.85, BLiMP 68.4%
Task 3 (Hybrid):     test PPL 23.086, BLiMP 0.647

This notebook required no retraining -- the AWD-LSTM checkpoint was
downloaded directly from this repository's GitHub Release.
